# Enhanced GAT + Quantum Grey Wolf Optimization for Fake News Detection

This notebook keeps the core model family as **Graph Attention Network (GAT) + quantum-behaved Grey Wolf Optimization (QGWO)** and extends the original script into a reproducible experiment workflow for the datasets under `F:\FAKENEWS`.

What is added:

- loaders for CSV, Excel graph tensors, JSONL fact-check data, and FakeNewsNet++ JSON files
- text-to-graph construction for datasets that do not already contain graph tensors
- QGWO hyperparameter search for the GAT classifier
- class-weighted training, early stopping, and repeatable train/validation/test splits
- article-ready metrics table with accuracy, macro metrics, weighted metrics, and ROC-AUC
- figures for class balance, QGWO convergence, training curves, confusion matrices, ROC curves, and experiment comparison

Suggested use:

1. Run once with `RUN_MODE = "fast"` to verify the full pipeline.
2. Switch to `RUN_MODE = "article"` for final Q1/IEEE-style experiments.
3. Report the held-out `test` split and, when possible, repeat with multiple seeds.

In [1]:
# Optional dependency cell.
# Uncomment only if your environment is missing packages.
#
# %pip install numpy pandas scikit-learn matplotlib openpyxl torch

In [2]:
from __future__ import annotations

import json
import math
import random
import re
import time
import warnings
from copy import deepcopy
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import Normalizer
from torch import Tensor, nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
BASE_DIR = Path(r"F:\FAKENEWS")
preferred_output_dir = BASE_DIR / "gat_qgwo_article_outputs"
try:
    preferred_output_dir.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR = preferred_output_dir
except PermissionError:
    OUTPUT_DIR = Path.cwd() / "gat_qgwo_article_outputs"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Permission denied for {preferred_output_dir}; using {OUTPUT_DIR}")

# Use "fast" for smoke tests. Use "article" for final tables and figures.
RUN_MODE = "fast"  # "fast" or "article"

if RUN_MODE == "fast":
    EXPERIMENT_SEEDS = [42]
    MAX_ROWS_PER_CLASS = 1200
    WOLVES = 4
    QGWO_ITERATIONS = 3
    CANDIDATE_EPOCHS = 3
    FINAL_EPOCHS = 12
    PATIENCE = 4
else:
    EXPERIMENT_SEEDS = [13, 42, 77, 2026, 3407]
    MAX_ROWS_PER_CLASS = None
    WOLVES = 8
    QGWO_ITERATIONS = 8
    CANDIDATE_EPOCHS = 8
    FINAL_EPOCHS = 60
    PATIENCE = 10

BATCH_SIZE = 96
NUM_THREADS = 2
COMPLEXITY_PENALTY = 1e-8
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.set_num_threads(max(1, NUM_THREADS))
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)
print(f"Device: {DEVICE}")
print(f"Output directory: {OUTPUT_DIR}")

Device: cpu
Output directory: F:\FAKENEWS\gat_qgwo_article_outputs


## Dataset Registry

Labels use `1 = fake/misinformation` and `0 = real/true`. For PolitiFact fact-check verdicts, `true` and `mostly-true` are mapped to real; `false`, `mostly-false`, and `pants-fire` are mapped to fake; `half-true` is dropped by default because it is ambiguous for binary fake-news classification.

In [3]:
@dataclass(frozen=True)
class SourceSpec:
    name: str
    path: Path
    kind: str
    forced_label: Optional[int] = None
    text_columns: Tuple[str, ...] = (
        "synthetic_misinformation",
        "synthetic misinformation",
        "title",
        "statement",
        "description",
        "news_text",
        "text",
    )
    combine_text: bool = False
    note: str = ""


SOURCE_SPECS: Dict[str, SourceSpec] = {
    "synthetic_hallucination": SourceSpec(
        "synthetic_hallucination",
        BASE_DIR / "synthetic-gpt-3.5-turbo_politifact_hallucination_processed.csv",
        "csv",
        note="Synthetic GPT-3.5 hallucination PolitiFact records",
    ),
    "synthetic_paraphrase": SourceSpec(
        "synthetic_paraphrase",
        BASE_DIR / "synthetic-gpt-3.5-turbo_politifact_paraphrase_generation_processed.csv",
        "csv",
        note="Synthetic GPT-3.5 paraphrase PolitiFact records",
    ),
    "synthetic_partially_arbitrary": SourceSpec(
        "synthetic_partially_arbitrary",
        BASE_DIR / "synthetic-gpt-3.5-turbo_politifact_partially_arbitrary_generation_politics_rumors_processed.csv",
        "csv",
        note="Synthetic politics rumor records",
    ),
    "gossipcop_root_fake": SourceSpec(
        "gossipcop_root_fake",
        BASE_DIR / "gossipcop_fake.csv",
        "csv",
        forced_label=1,
        note="FakeNewsNet GossipCop fake CSV",
    ),
    "gossipcop_root_real": SourceSpec(
        "gossipcop_root_real",
        BASE_DIR / "gossipcop_real.csv",
        "csv",
        forced_label=0,
        note="FakeNewsNet GossipCop real CSV",
    ),
    "gossipcop_new_fake": SourceSpec(
        "gossipcop_new_fake",
        BASE_DIR / "New folder" / "dataset" / "gossipcop_fake.csv",
        "csv",
        forced_label=1,
        note="Nested FakeNewsNet GossipCop fake CSV",
    ),
    "gossipcop_new_real": SourceSpec(
        "gossipcop_new_real",
        BASE_DIR / "New folder" / "dataset" / "gossipcop_real.csv",
        "csv",
        forced_label=0,
        note="Nested FakeNewsNet GossipCop real CSV",
    ),
    "politifact_new_fake": SourceSpec(
        "politifact_new_fake",
        BASE_DIR / "New folder" / "dataset" / "politifact_fake.csv",
        "csv",
        forced_label=1,
        note="Nested FakeNewsNet PolitiFact fake CSV",
    ),
    "politifact_new_real": SourceSpec(
        "politifact_new_real",
        BASE_DIR / "New folder" / "dataset" / "politifact_real.csv",
        "csv",
        forced_label=0,
        note="Nested FakeNewsNet PolitiFact real CSV",
    ),
    "politifact_factcheck": SourceSpec(
        "politifact_factcheck",
        BASE_DIR / "politifact_factcheck_data.json",
        "factcheck_jsonl",
        text_columns=("statement",),
        note="PolitiFact verdict JSONL mapped to binary labels",
    ),
    "gossipcoppp_hf": SourceSpec(
        "gossipcoppp_hf",
        BASE_DIR / "Data" / "GossipCop++" / "HF.json",
        "json_object",
        forced_label=1,
        combine_text=True,
        note="GossipCop++ human-written fake",
    ),
    "gossipcoppp_hr": SourceSpec(
        "gossipcoppp_hr",
        BASE_DIR / "Data" / "GossipCop++" / "HR.json",
        "json_object",
        forced_label=0,
        combine_text=True,
        note="GossipCop++ human-written real",
    ),
    "gossipcoppp_mf": SourceSpec(
        "gossipcoppp_mf",
        BASE_DIR / "Data" / "GossipCop++" / "MF.json",
        "json_object",
        forced_label=1,
        combine_text=True,
        note="GossipCop++ machine-generated fake",
    ),
    "gossipcoppp_mr": SourceSpec(
        "gossipcoppp_mr",
        BASE_DIR / "Data" / "GossipCop++" / "MR.json",
        "json_object",
        forced_label=0,
        combine_text=True,
        note="GossipCop++ machine-generated real",
    ),
    "politifactpp_hf": SourceSpec(
        "politifactpp_hf",
        BASE_DIR / "Data" / "PolitiFact++" / "HF.json",
        "json_object",
        forced_label=1,
        combine_text=True,
        note="PolitiFact++ human-written fake",
    ),
    "politifactpp_hr": SourceSpec(
        "politifactpp_hr",
        BASE_DIR / "Data" / "PolitiFact++" / "HR.json",
        "json_object",
        forced_label=0,
        combine_text=True,
        note="PolitiFact++ human-written real",
    ),
    "politifactpp_mf": SourceSpec(
        "politifactpp_mf",
        BASE_DIR / "Data" / "PolitiFact++" / "MF.json",
        "json_object",
        forced_label=1,
        combine_text=True,
        note="PolitiFact++ machine-generated fake",
    ),
    "politifactpp_mr": SourceSpec(
        "politifactpp_mr",
        BASE_DIR / "Data" / "PolitiFact++" / "MR.json",
        "json_object",
        forced_label=0,
        combine_text=True,
        note="PolitiFact++ machine-generated real",
    ),
}


EXPERIMENTS: Dict[str, Dict[str, Any]] = {
    "prepared_excel_graphs": {
        "kind": "excel_graph",
        "path": BASE_DIR / "fake_news.xlsx",
        "note": "Native graph tensors from the Excel Graphs sheet",
    },
    "synthetic_hallucination": {"kind": "text", "sources": ["synthetic_hallucination"]},
    "synthetic_paraphrase": {"kind": "text", "sources": ["synthetic_paraphrase"]},
    "synthetic_partially_arbitrary": {"kind": "text", "sources": ["synthetic_partially_arbitrary"]},
    "synthetic_politifact_all": {
        "kind": "text",
        "sources": [
            "synthetic_hallucination",
            "synthetic_paraphrase",
            "synthetic_partially_arbitrary",
        ],
    },
    "gossipcop_root": {
        "kind": "text",
        "sources": ["gossipcop_root_fake", "gossipcop_root_real"],
    },
    "gossipcop_new_folder": {
        "kind": "text",
        "sources": ["gossipcop_new_fake", "gossipcop_new_real"],
    },
    "politifact_new_folder": {
        "kind": "text",
        "sources": ["politifact_new_fake", "politifact_new_real"],
    },
    "fakenewsnet_gossipcop_combined": {
        "kind": "text",
        "sources": [
            "gossipcop_root_fake",
            "gossipcop_root_real",
            "gossipcop_new_fake",
            "gossipcop_new_real",
        ],
    },
    "fakenewsnet_gossipcop_politifact_combined": {
        "kind": "text",
        "sources": [
            "gossipcop_root_fake",
            "gossipcop_root_real",
            "gossipcop_new_fake",
            "gossipcop_new_real",
            "politifact_new_fake",
            "politifact_new_real",
        ],
    },
    "factcheck_binary": {"kind": "text", "sources": ["politifact_factcheck"]},
    "synthetic_plus_factcheck": {
        "kind": "text",
        "sources": [
            "synthetic_hallucination",
            "synthetic_paraphrase",
            "synthetic_partially_arbitrary",
            "politifact_factcheck",
        ],
    },
    "gossipcop_plus_plus_human": {
        "kind": "text",
        "sources": ["gossipcoppp_hf", "gossipcoppp_hr"],
    },
    "gossipcop_plus_plus_machine": {
        "kind": "text",
        "sources": ["gossipcoppp_mf", "gossipcoppp_mr"],
    },
    "gossipcop_plus_plus_all": {
        "kind": "text",
        "sources": ["gossipcoppp_hf", "gossipcoppp_hr", "gossipcoppp_mf", "gossipcoppp_mr"],
    },
    "politifact_plus_plus_human": {
        "kind": "text",
        "sources": ["politifactpp_hf", "politifactpp_hr"],
    },
    "politifact_plus_plus_machine": {
        "kind": "text",
        "sources": ["politifactpp_mf", "politifactpp_mr"],
    },
    "politifact_plus_plus_all": {
        "kind": "text",
        "sources": ["politifactpp_hf", "politifactpp_hr", "politifactpp_mf", "politifactpp_mr"],
    },
    "all_text_sources": {"kind": "text", "sources": list(SOURCE_SPECS.keys())},
}

if RUN_MODE == "fast":
    SELECTED_EXPERIMENTS = [
        "prepared_excel_graphs",
        "synthetic_politifact_all",
        "gossipcop_root",
        "politifact_new_folder",
        "factcheck_binary",
        "gossipcop_plus_plus_all",
        "politifact_plus_plus_all",
        "all_text_sources",
    ]
else:
    SELECTED_EXPERIMENTS = list(EXPERIMENTS.keys())

print(f"Registered sources: {len(SOURCE_SPECS)}")
print(f"Selected experiments: {len(SELECTED_EXPERIMENTS)}")

Registered sources: 18
Selected experiments: 8


In [4]:
TRUE_VERDICTS = {"true", "mostly-true"}
FALSE_VERDICTS = {"false", "mostly-false", "pants-fire"}


def clean_text(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, float) and np.isnan(value):
        return ""
    text = str(value)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def normalize_label(value: Any) -> Optional[int]:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    if isinstance(value, (int, np.integer)):
        return int(value)
    if isinstance(value, (float, np.floating)) and float(value).is_integer():
        return int(value)
    lowered = str(value).strip().lower()
    if lowered in {"1", "fake", "false", "misinformation", "rumor", "rumour"}:
        return 1
    if lowered in {"0", "real", "true", "reliable"}:
        return 0
    if lowered in TRUE_VERDICTS:
        return 0
    if lowered in FALSE_VERDICTS:
        return 1
    return None


def choose_text(row: pd.Series, columns: Sequence[str], combine: bool) -> str:
    pieces: List[str] = []
    for column in columns:
        if column in row.index:
            value = clean_text(row[column])
            if value:
                if not combine:
                    return value
                pieces.append(value)
    return " ".join(pieces)


def _id_column(columns: Sequence[str]) -> Optional[str]:
    for column in ("id", "news_id", "graph_id", "statement_id"):
        if column in columns:
            return column
    return None


def load_csv_source(spec: SourceSpec) -> pd.DataFrame:
    frame = pd.read_csv(spec.path)
    id_col = _id_column(frame.columns)
    if spec.forced_label is None:
        label_col = next((c for c in ("label", "target", "class", "verdict") if c in frame.columns), None)
        if label_col is None:
            raise ValueError(f"No label column found for {spec.name}")
        labels = frame[label_col].map(normalize_label)
    else:
        labels = pd.Series(spec.forced_label, index=frame.index)
    text = frame.apply(lambda row: choose_text(row, spec.text_columns, spec.combine_text), axis=1)
    out = pd.DataFrame(
        {
            "source_id": frame[id_col].astype(str) if id_col else [f"{spec.name}-{i}" for i in range(len(frame))],
            "text": text,
            "label": labels,
            "source": spec.name,
            "source_note": spec.note,
        }
    )
    return out


def load_factcheck_jsonl(spec: SourceSpec) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    with open(spec.path, "r", encoding="utf-8") as file:
        for line_no, line in enumerate(file, start=1):
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            verdict = clean_text(record.get("verdict")).lower()
            if verdict in TRUE_VERDICTS:
                label = 0
            elif verdict in FALSE_VERDICTS:
                label = 1
            else:
                continue
            text = choose_text(pd.Series(record), spec.text_columns, spec.combine_text)
            rows.append(
                {
                    "source_id": clean_text(record.get("factcheck_analysis_link")) or f"{spec.name}-{line_no}",
                    "text": text,
                    "label": label,
                    "source": spec.name,
                    "source_note": f"{spec.note}; verdict={verdict}",
                }
            )
    return pd.DataFrame(rows)


def load_json_object_source(spec: SourceSpec) -> pd.DataFrame:
    with open(spec.path, "r", encoding="utf-8") as file:
        data = json.load(file)
    records = data.values() if isinstance(data, dict) else data
    rows: List[Dict[str, Any]] = []
    for index, record in enumerate(records):
        series = pd.Series(record)
        label = spec.forced_label if spec.forced_label is not None else normalize_label(record.get("label"))
        rows.append(
            {
                "source_id": clean_text(record.get("id")) or f"{spec.name}-{index}",
                "text": choose_text(series, spec.text_columns, spec.combine_text),
                "label": label,
                "source": spec.name,
                "source_note": spec.note,
            }
        )
    return pd.DataFrame(rows)


def load_source(name: str) -> pd.DataFrame:
    spec = SOURCE_SPECS[name]
    if not spec.path.exists():
        print(f"Missing source: {spec.path}")
        return pd.DataFrame(columns=["source_id", "text", "label", "source", "source_note"])
    if spec.kind == "csv":
        frame = load_csv_source(spec)
    elif spec.kind == "factcheck_jsonl":
        frame = load_factcheck_jsonl(spec)
    elif spec.kind == "json_object":
        frame = load_json_object_source(spec)
    else:
        raise ValueError(f"Unknown source kind: {spec.kind}")
    frame["label"] = frame["label"].map(normalize_label)
    frame["text"] = frame["text"].map(clean_text)
    frame = frame.dropna(subset=["label"])
    frame = frame[frame["text"].str.len() > 0].copy()
    frame["label"] = frame["label"].astype(int)
    return frame.reset_index(drop=True)


def load_sources(names: Sequence[str]) -> pd.DataFrame:
    frames = [load_source(name) for name in names]
    frames = [frame for frame in frames if len(frame)]
    if not frames:
        return pd.DataFrame(columns=["source_id", "text", "label", "source", "source_note"])
    combined = pd.concat(frames, ignore_index=True)
    combined["text_key"] = (
        combined["text"]
        .str.lower()
        .str.replace(r"\W+", " ", regex=True)
        .str.strip()
        .str.slice(0, 900)
    )
    combined = combined.drop_duplicates("text_key", keep="first")
    return combined.drop(columns=["text_key"]).reset_index(drop=True)


def sample_per_class(frame: pd.DataFrame, max_rows_per_class: Optional[int], seed: int) -> pd.DataFrame:
    if max_rows_per_class is None:
        return frame.reset_index(drop=True)
    sampled = []
    for label, group in frame.groupby("label", sort=True):
        n = min(len(group), max_rows_per_class)
        sampled.append(group.sample(n=n, random_state=seed))
    return pd.concat(sampled, ignore_index=True).sample(frac=1.0, random_state=seed).reset_index(drop=True)


source_summary_rows = []
for source_name in SOURCE_SPECS:
    try:
        frame = load_source(source_name)
        counts = frame["label"].value_counts().to_dict() if len(frame) else {}
        source_summary_rows.append(
            {
                "source": source_name,
                "rows": len(frame),
                "real_0": counts.get(0, 0),
                "fake_1": counts.get(1, 0),
                "path": str(SOURCE_SPECS[source_name].path),
            }
        )
    except Exception as exc:
        source_summary_rows.append(
            {
                "source": source_name,
                "rows": 0,
                "real_0": 0,
                "fake_1": 0,
                "path": str(SOURCE_SPECS[source_name].path),
                "error": str(exc),
            }
        )

source_summary = pd.DataFrame(source_summary_rows)
source_summary.to_csv(OUTPUT_DIR / "source_summary.csv", index=False)
display(source_summary)

,source,rows,real_0,fake_1,path
0,synthetic_hallucination,100,0,100,F:\FAKENEWS\synthetic-gpt-3.5-turbo_politifact...
1,synthetic_paraphrase,415,145,270,F:\FAKENEWS\synthetic-gpt-3.5-turbo_politifact...
2,synthetic_partially_arbitrary,100,0,100,F:\FAKENEWS\synthetic-gpt-3.5-turbo_politifact...
3,gossipcop_root_fake,5323,0,5323,F:\FAKENEWS\gossipcop_fake.csv
4,gossipcop_root_real,16817,16817,0,F:\FAKENEWS\gossipcop_real.csv
5,gossipcop_new_fake,5323,0,5323,F:\FAKENEWS\New folder\dataset\gossipcop_fake.csv
6,gossipcop_new_real,16817,16817,0,F:\FAKENEWS\New folder\dataset\gossipcop_real.csv
7,politifact_new_fake,432,0,432,F:\FAKENEWS\New folder\dataset\politifact_fake...
8,politifact_new_real,624,624,0,F:\FAKENEWS\New folder\dataset\politifact_real...
9,politifact_factcheck,17555,5795,11760,F:\FAKENEWS\politifact_factcheck_data.json


## Graph Tensor Construction

The Excel file already contains graph tensors. Other datasets are transformed into document graphs:

- each article is split into sentence/chunk nodes
- node features are TF-IDF vectors projected with TruncatedSVD
- edges connect self-loops, neighboring chunks, and top-k semantically similar chunks
- the GAT receives fixed-size padded tensors plus a node mask

This preserves the GAT model type while making all listed datasets usable in one comparison pipeline.

In [5]:
@dataclass(frozen=True)
class TextGraphConfig:
    max_nodes: int = 20
    words_per_node: int = 70
    feature_dim: int = 64
    max_tfidf_features: int = 30000
    top_k: int = 3
    min_df: int = 2
    ngram_range: Tuple[int, int] = (1, 2)


TEXT_GRAPH_CONFIG = TextGraphConfig()


@dataclass
class GraphBundle:
    name: str
    x: Tensor
    adj: Tensor
    mask: Tensor
    y: Tensor
    split: np.ndarray
    ids: np.ndarray
    metadata: pd.DataFrame
    note: str = ""

    @property
    def num_features(self) -> int:
        return int(self.x.shape[-1])

    @property
    def num_classes(self) -> int:
        return int(torch.unique(self.y).numel())

    def indices(self, split_name: str) -> Tensor:
        return torch.tensor(np.where(self.split == split_name)[0], dtype=torch.long)


class ExcelGraphTensors:
    def __init__(self, excel_path: Path, cache_path: Optional[Path] = None):
        self.excel_path = Path(excel_path)
        if not self.excel_path.exists():
            raise FileNotFoundError(f"Excel dataset not found: {self.excel_path}")

        if cache_path is not None and cache_path.exists():
            cached = torch.load(cache_path, map_location="cpu", weights_only=False)
            self.x = cached["x"]
            self.adj = cached["adj"]
            self.y = cached["y"]
            self.split = cached["split"]
            self.graph_ids = cached["graph_ids"]
            self.mask = cached.get("mask", torch.ones(self.x.shape[:2], dtype=torch.bool))
            self._validate()
            return

        frame = pd.read_excel(self.excel_path, sheet_name="Graphs")
        required = {"graph_id", "label", "split", "node_features_json", "edge_index_json"}
        missing = required.difference(frame.columns)
        if missing:
            raise ValueError(f"Missing columns in Excel graph sheet: {sorted(missing)}")

        features: List[np.ndarray] = []
        adjacency: List[np.ndarray] = []
        labels: List[int] = []
        splits: List[str] = []
        graph_ids: List[str] = []

        for row in frame.itertuples(index=False):
            x = np.asarray(json.loads(row.node_features_json), dtype=np.float32)
            edge_index = np.asarray(json.loads(row.edge_index_json), dtype=np.int64)
            if x.ndim != 2:
                raise ValueError(f"Graph {row.graph_id}: features must be 2D")
            if edge_index.ndim != 2 or edge_index.shape[0] != 2:
                raise ValueError(f"Graph {row.graph_id}: edge_index must have shape [2, E]")
            num_nodes = x.shape[0]
            adj = np.zeros((num_nodes, num_nodes), dtype=np.bool_)
            sources = edge_index[0]
            targets = edge_index[1]
            adj[targets, sources] = True
            np.fill_diagonal(adj, True)
            features.append(x)
            adjacency.append(adj)
            labels.append(int(row.label))
            splits.append(clean_text(row.split).lower())
            graph_ids.append(clean_text(row.graph_id))

        self.x = torch.from_numpy(np.stack(features))
        self.adj = torch.from_numpy(np.stack(adjacency))
        self.mask = torch.ones(self.x.shape[:2], dtype=torch.bool)
        self.y = torch.tensor(labels, dtype=torch.long)
        self.split = np.asarray(splits)
        self.graph_ids = np.asarray(graph_ids)
        self._validate()

        if cache_path is not None:
            cache_path.parent.mkdir(parents=True, exist_ok=True)
            torch.save(
                {
                    "x": self.x,
                    "adj": self.adj,
                    "mask": self.mask,
                    "y": self.y,
                    "split": self.split,
                    "graph_ids": self.graph_ids,
                },
                cache_path,
            )

    def _validate(self) -> None:
        if self.x.ndim != 3:
            raise ValueError(f"Expected x=[G,N,F], got {tuple(self.x.shape)}")
        if self.adj.shape[:2] != self.x.shape[:2] or self.adj.shape[2] != self.x.shape[1]:
            raise ValueError("Adjacency shape is inconsistent with features")
        if self.mask.shape != self.x.shape[:2]:
            raise ValueError("Mask shape is inconsistent with features")

    def to_bundle(self, name: str) -> GraphBundle:
        metadata = pd.DataFrame({"source_id": self.graph_ids, "source": name, "label": self.y.numpy()})
        return GraphBundle(
            name=name,
            x=self.x.float(),
            adj=self.adj.bool(),
            mask=self.mask.bool(),
            y=self.y.long(),
            split=self.split,
            ids=self.graph_ids,
            metadata=metadata,
            note="Excel Graphs sheet",
        )

In [6]:
SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")


def split_text_to_nodes(text: str, max_nodes: int, words_per_node: int) -> List[str]:
    text = clean_text(text)
    if not text:
        return ["empty"]
    raw_sentences = [s.strip() for s in SENTENCE_SPLIT_RE.split(text) if s.strip()]
    if not raw_sentences:
        raw_sentences = [text]

    nodes: List[str] = []
    for sentence in raw_sentences:
        words = sentence.split()
        if len(words) <= words_per_node:
            nodes.append(sentence)
        else:
            for start in range(0, len(words), words_per_node):
                nodes.append(" ".join(words[start : start + words_per_node]))
        if len(nodes) >= max_nodes:
            break
    return nodes[:max_nodes] or ["empty"]


def add_stratified_splits(
    frame: pd.DataFrame,
    seed: int,
    train_size: float = 0.70,
    validation_size: float = 0.15,
    test_size: float = 0.15,
) -> pd.DataFrame:
    if not math.isclose(train_size + validation_size + test_size, 1.0):
        raise ValueError("Split sizes must sum to 1")
    counts = frame["label"].value_counts()
    if len(counts) < 2:
        raise ValueError("Need at least two classes for binary fake-news detection")
    if counts.min() < 3:
        raise ValueError("Each class needs at least three examples for train/validation/test")

    train_frame, temp_frame = train_test_split(
        frame,
        test_size=(validation_size + test_size),
        stratify=frame["label"],
        random_state=seed,
    )
    relative_test = test_size / (validation_size + test_size)
    validation_frame, test_frame = train_test_split(
        temp_frame,
        test_size=relative_test,
        stratify=temp_frame["label"],
        random_state=seed,
    )
    train_frame = train_frame.copy()
    validation_frame = validation_frame.copy()
    test_frame = test_frame.copy()
    train_frame["split"] = "train"
    validation_frame["split"] = "validation"
    test_frame["split"] = "test"
    return pd.concat([train_frame, validation_frame, test_frame], ignore_index=True)


def _l2_normalize(array: np.ndarray) -> np.ndarray:
    denom = np.linalg.norm(array, axis=1, keepdims=True)
    denom[denom == 0.0] = 1.0
    return array / denom


def fit_text_encoder(
    train_node_lists: Sequence[Sequence[str]],
    config: TextGraphConfig,
    seed: int,
) -> Dict[str, Any]:
    train_nodes = [node for nodes in train_node_lists for node in nodes]
    if not train_nodes:
        train_nodes = ["empty"]
    vectorizer = TfidfVectorizer(
        max_features=config.max_tfidf_features,
        min_df=config.min_df,
        max_df=0.95,
        stop_words="english",
        ngram_range=config.ngram_range,
        sublinear_tf=True,
    )
    try:
        tfidf = vectorizer.fit_transform(train_nodes)
    except ValueError:
        vectorizer = TfidfVectorizer(
            max_features=config.max_tfidf_features,
            min_df=1,
            max_df=1.0,
            stop_words=None,
            ngram_range=(1, 1),
            sublinear_tf=True,
        )
        tfidf = vectorizer.fit_transform(train_nodes)

    svd = None
    normalizer = None
    if tfidf.shape[1] > config.feature_dim + 1:
        n_components = min(config.feature_dim, tfidf.shape[1] - 1)
        svd = TruncatedSVD(n_components=n_components, random_state=seed)
        svd.fit(tfidf)
        normalizer = Normalizer(copy=False)

    return {"vectorizer": vectorizer, "svd": svd, "normalizer": normalizer}


def transform_flat_nodes(
    flat_nodes: Sequence[str],
    encoder: Dict[str, Any],
    feature_dim: int,
    batch_size: int = 20000,
) -> np.ndarray:
    if not flat_nodes:
        return np.zeros((0, feature_dim), dtype=np.float32)

    vectorizer = encoder["vectorizer"]
    svd = encoder["svd"]
    normalizer = encoder["normalizer"]
    chunks: List[np.ndarray] = []
    for start in range(0, len(flat_nodes), batch_size):
        node_batch = flat_nodes[start : start + batch_size]
        tfidf = vectorizer.transform(node_batch)
        if svd is not None:
            array = svd.transform(tfidf)
            if normalizer is not None:
                array = normalizer.transform(array)
        else:
            array = tfidf.toarray()
            array = _l2_normalize(array)
        if array.shape[1] < feature_dim:
            pad = np.zeros((array.shape[0], feature_dim - array.shape[1]), dtype=np.float32)
            array = np.hstack([array.astype(np.float32), pad])
        elif array.shape[1] > feature_dim:
            array = array[:, :feature_dim]
        chunks.append(array.astype(np.float32))
    return np.vstack(chunks)


def build_adjacency(node_features: np.ndarray, valid_nodes: int, max_nodes: int, top_k: int) -> np.ndarray:
    adj = np.zeros((max_nodes, max_nodes), dtype=np.bool_)
    if valid_nodes <= 0:
        adj[0, 0] = True
        return adj
    valid_nodes = min(valid_nodes, max_nodes)
    for i in range(max_nodes):
        adj[i, i] = True
    for i in range(valid_nodes - 1):
        adj[i, i + 1] = True
        adj[i + 1, i] = True
    if valid_nodes > 1 and top_k > 0:
        features = _l2_normalize(node_features[:valid_nodes])
        similarity = features @ features.T
        np.fill_diagonal(similarity, -np.inf)
        k = min(top_k, valid_nodes - 1)
        for i in range(valid_nodes):
            neighbors = np.argpartition(-similarity[i], kth=k - 1)[:k]
            adj[i, neighbors] = True
            adj[neighbors, i] = True
    return adj


def make_text_graph_bundle(
    name: str,
    frame: pd.DataFrame,
    config: TextGraphConfig,
    seed: int,
    note: str,
) -> GraphBundle:
    frame = add_stratified_splits(frame, seed=seed)
    node_lists = [
        split_text_to_nodes(text, max_nodes=config.max_nodes, words_per_node=config.words_per_node)
        for text in frame["text"].tolist()
    ]
    train_node_lists = [nodes for nodes, split in zip(node_lists, frame["split"]) if split == "train"]
    encoder = fit_text_encoder(train_node_lists, config=config, seed=seed)

    flat_nodes: List[str] = []
    spans: List[Tuple[int, int]] = []
    cursor = 0
    for nodes in node_lists:
        spans.append((cursor, cursor + len(nodes)))
        flat_nodes.extend(nodes)
        cursor += len(nodes)
    flat_features = transform_flat_nodes(flat_nodes, encoder=encoder, feature_dim=config.feature_dim)

    n_graphs = len(frame)
    x = np.zeros((n_graphs, config.max_nodes, config.feature_dim), dtype=np.float32)
    adj = np.zeros((n_graphs, config.max_nodes, config.max_nodes), dtype=np.bool_)
    mask = np.zeros((n_graphs, config.max_nodes), dtype=np.bool_)

    for graph_index, (start, end) in enumerate(spans):
        features = flat_features[start:end]
        valid = min(len(features), config.max_nodes)
        if valid == 0:
            valid = 1
        x[graph_index, :valid] = features[:valid]
        mask[graph_index, :valid] = True
        adj[graph_index] = build_adjacency(x[graph_index], valid, config.max_nodes, config.top_k)

    ids = frame["source_id"].astype(str).to_numpy()
    return GraphBundle(
        name=name,
        x=torch.from_numpy(x),
        adj=torch.from_numpy(adj),
        mask=torch.from_numpy(mask),
        y=torch.tensor(frame["label"].astype(int).to_numpy(), dtype=torch.long),
        split=frame["split"].to_numpy(),
        ids=ids,
        metadata=frame.reset_index(drop=True),
        note=note,
    )

## Dense Multi-Head GAT

This is the same dense, multi-head GAT family as the original code, with a masked graph readout so padded text-graph nodes do not affect graph-level predictions.

In [7]:
@dataclass(frozen=True)
class GATConfig:
    hidden_channels: int
    num_heads: int
    num_layers: int
    dropout: float
    learning_rate: float
    weight_decay: float
    label_smoothing: float


@dataclass
class TrainResult:
    model: nn.Module
    history: List[Dict[str, float]]
    best_epoch: int
    best_validation_f1: float


class DenseMultiHeadGATLayer(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, heads: int, dropout: float):
        super().__init__()
        self.out_channels = out_channels
        self.heads = heads
        self.dropout = dropout
        self.projection = nn.Linear(in_channels, out_channels * heads, bias=False)
        self.attention_source = nn.Parameter(torch.empty(heads, out_channels))
        self.attention_target = nn.Parameter(torch.empty(heads, out_channels))
        self.bias = nn.Parameter(torch.zeros(out_channels * heads))
        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.xavier_uniform_(self.projection.weight)
        nn.init.xavier_uniform_(self.attention_source)
        nn.init.xavier_uniform_(self.attention_target)
        nn.init.zeros_(self.bias)

    def forward(self, x: Tensor, adj: Tensor) -> Tensor:
        batch_size, num_nodes, _ = x.shape
        projected = self.projection(x).view(batch_size, num_nodes, self.heads, self.out_channels)
        projected = projected.permute(0, 2, 1, 3)

        source_scores = (projected * self.attention_source[None, :, None, :]).sum(dim=-1)
        target_scores = (projected * self.attention_target[None, :, None, :]).sum(dim=-1)
        logits = F.leaky_relu(target_scores.unsqueeze(-1) + source_scores.unsqueeze(-2), negative_slope=0.2)
        logits = logits.masked_fill(~adj[:, None, :, :], -1e9)

        attention = torch.softmax(logits, dim=-1)
        attention = F.dropout(attention, p=self.dropout, training=self.training)
        output = torch.matmul(attention, projected)
        output = output.permute(0, 2, 1, 3).contiguous()
        output = output.view(batch_size, num_nodes, self.heads * self.out_channels)
        return output + self.bias


class JKDeepGATNet(nn.Module):
    def __init__(
        self,
        num_features: int,
        hidden_channels: int,
        num_heads: int,
        num_layers: int,
        dropout: float,
        num_classes: int = 2,
    ):
        super().__init__()
        hidden_dim = hidden_channels * num_heads
        self.dropout = dropout
        self.input_projection = nn.Linear(num_features, hidden_dim)
        self.layers = nn.ModuleList()
        self.normalizations = nn.ModuleList()
        for _ in range(num_layers):
            self.layers.append(
                DenseMultiHeadGATLayer(
                    in_channels=hidden_dim,
                    out_channels=hidden_channels,
                    heads=num_heads,
                    dropout=dropout,
                )
            )
            self.normalizations.append(nn.LayerNorm(hidden_dim))
        readout_dim = (num_layers + 1) * hidden_dim * 2
        self.classifier = nn.Sequential(
            nn.Linear(readout_dim, 96),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(96, num_classes),
        )

    @staticmethod
    def graph_readout(x: Tensor, mask: Tensor) -> Tensor:
        mask_float = mask.unsqueeze(-1).float()
        denom = mask_float.sum(dim=1).clamp_min(1.0)
        mean_pool = (x * mask_float).sum(dim=1) / denom
        masked_x = x.masked_fill(~mask.unsqueeze(-1), -1e9)
        max_pool = masked_x.max(dim=1).values
        max_pool = torch.where(torch.isfinite(max_pool), max_pool, torch.zeros_like(max_pool))
        return torch.cat([mean_pool, max_pool], dim=1)

    def forward(self, x: Tensor, adj: Tensor, mask: Tensor) -> Tensor:
        x = F.elu(self.input_projection(x))
        x = x * mask.unsqueeze(-1).float()
        readouts = [self.graph_readout(x, mask)]
        for layer, normalization in zip(self.layers, self.normalizations):
            x = F.elu(normalization(layer(x, adj)))
            x = x * mask.unsqueeze(-1).float()
            readouts.append(self.graph_readout(x, mask))
        graph_embedding = torch.cat(readouts, dim=1)
        return self.classifier(graph_embedding)


def count_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters())

In [8]:
def make_loader(bundle: GraphBundle, indices: Tensor, batch_size: int, shuffle: bool, seed: int) -> DataLoader:
    dataset = TensorDataset(
        bundle.x[indices],
        bundle.adj[indices],
        bundle.mask[indices],
        bundle.y[indices],
        indices,
    )
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator if shuffle else None,
        num_workers=0,
    )


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, y_prob: np.ndarray) -> Dict[str, float]:
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }
    try:
        metrics["roc_auc"] = roc_auc_score(y_true, y_prob)
    except ValueError:
        metrics["roc_auc"] = np.nan
    return metrics


def evaluate(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    model.eval()
    labels: List[int] = []
    predictions: List[int] = []
    probabilities: List[float] = []
    row_indices: List[int] = []

    with torch.no_grad():
        for x, adj, mask, y, batch_indices in loader:
            x = x.to(device).float()
            adj = adj.to(device).bool()
            mask = mask.to(device).bool()
            y = y.to(device)
            logits = model(x, adj, mask)
            probability = torch.softmax(logits, dim=1)[:, 1]
            prediction = logits.argmax(dim=1)
            labels.extend(y.cpu().numpy().tolist())
            predictions.extend(prediction.cpu().numpy().tolist())
            probabilities.extend(probability.cpu().numpy().tolist())
            row_indices.extend(batch_indices.cpu().numpy().tolist())

    y_true = np.asarray(labels)
    y_pred = np.asarray(predictions)
    y_prob = np.asarray(probabilities)
    ids = np.asarray(row_indices)
    return compute_metrics(y_true, y_pred, y_prob), y_true, y_pred, y_prob, ids


def class_weight_tensor(y: Tensor, device: torch.device) -> Tensor:
    labels = y.cpu().numpy()
    counts = np.bincount(labels, minlength=2).astype(np.float32)
    counts[counts == 0] = 1.0
    weights = counts.sum() / (len(counts) * counts)
    return torch.tensor(weights, dtype=torch.float32, device=device)


def train_model(
    model: nn.Module,
    bundle: GraphBundle,
    train_loader: DataLoader,
    validation_loader: DataLoader,
    train_indices: Tensor,
    device: torch.device,
    epochs: int,
    learning_rate: float,
    weight_decay: float,
    patience: int,
    label_smoothing: float,
) -> TrainResult:
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    weights = class_weight_tensor(bundle.y[train_indices], device=device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=label_smoothing)

    best_state = deepcopy(model.state_dict())
    best_validation_f1 = -math.inf
    best_epoch = 0
    stale_epochs = 0
    history: List[Dict[str, float]] = []

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        samples = 0
        for x, adj, mask, y, _ in train_loader:
            x = x.to(device).float()
            adj = adj.to(device).bool()
            mask = mask.to(device).bool()
            y = y.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x, adj, mask)
            loss = criterion(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            running_loss += float(loss.item()) * y.shape[0]
            samples += int(y.shape[0])

        validation_metrics, _, _, _, _ = evaluate(model, validation_loader, device)
        validation_f1 = validation_metrics["f1_macro"]
        history.append(
            {
                "epoch": epoch,
                "train_loss": running_loss / max(samples, 1),
                "validation_accuracy": validation_metrics["accuracy"],
                "validation_f1_macro": validation_f1,
                "validation_roc_auc": validation_metrics["roc_auc"],
            }
        )

        if validation_f1 > best_validation_f1 + 1e-8:
            best_validation_f1 = validation_f1
            best_epoch = epoch
            best_state = deepcopy(model.state_dict())
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= patience:
                break

    model.load_state_dict(best_state)
    return TrainResult(model=model, history=history, best_epoch=best_epoch, best_validation_f1=best_validation_f1)

## Quantum Grey Wolf Optimization

QGWO tunes the GAT hyperparameters. The core optimizer is a quantum-inspired stochastic optimizer, not quantum hardware.

In [9]:
def decode_position(position: np.ndarray) -> GATConfig:
    hidden_options = [8, 12, 16, 24, 32]
    head_options = [2, 3, 4, 6]
    layer_options = [1, 2, 3]

    def option(values: Sequence[int], value: float) -> int:
        index = min(int(value * len(values)), len(values) - 1)
        return int(values[index])

    hidden_channels = option(hidden_options, position[0])
    num_heads = option(head_options, position[1])
    num_layers = option(layer_options, position[2])
    dropout = 0.05 + 0.50 * float(position[3])
    learning_rate = 10 ** (-4.2 + 2.2 * float(position[4]))
    weight_decay = 10 ** (-6.5 + 4.0 * float(position[5]))
    label_smoothing = 0.08 * float(position[6])

    return GATConfig(
        hidden_channels=hidden_channels,
        num_heads=num_heads,
        num_layers=num_layers,
        dropout=round(dropout, 5),
        learning_rate=float(learning_rate),
        weight_decay=float(weight_decay),
        label_smoothing=round(label_smoothing, 5),
    )


class QuantumGreyWolfOptimizer:
    def __init__(
        self,
        wolves: int,
        iterations: int,
        seed: int,
        dimensions: int = 7,
        beta_max: float = 1.0,
        beta_min: float = 0.45,
        tunnelling_probability: float = 0.12,
    ):
        if wolves < 3:
            raise ValueError("QGWO requires at least three wolves")
        if iterations < 1:
            raise ValueError("iterations must be at least one")
        self.wolves = wolves
        self.iterations = iterations
        self.dimensions = dimensions
        self.rng = np.random.default_rng(seed)
        self.beta_max = beta_max
        self.beta_min = beta_min
        self.tunnelling_probability = tunnelling_probability
        self.cache: Dict[GATConfig, Dict[str, float]] = {}
        self.search_log: List[Dict[str, float]] = []

    def _standard_gwo_proposal(self, current: np.ndarray, leader: np.ndarray, a: float) -> np.ndarray:
        r1 = self.rng.random(current.shape)
        r2 = self.rng.random(current.shape)
        A = 2.0 * a * r1 - a
        C = 2.0 * r2
        distance = np.abs(C * leader - current)
        return leader - A * distance

    def _quantum_update(
        self,
        positions: np.ndarray,
        alpha: np.ndarray,
        beta: np.ndarray,
        delta: np.ndarray,
        iteration: int,
    ) -> np.ndarray:
        progress = iteration / max(self.iterations - 1, 1)
        a = 2.0 * (1.0 - progress)
        beta_q = self.beta_max - (self.beta_max - self.beta_min) * progress
        leader_centroid = (alpha + beta + delta) / 3.0

        updated_positions = []
        for current in positions:
            proposal_alpha = self._standard_gwo_proposal(current, alpha, a)
            proposal_beta = self._standard_gwo_proposal(current, beta, a)
            proposal_delta = self._standard_gwo_proposal(current, delta, a)
            attractor = (proposal_alpha + proposal_beta + proposal_delta) / 3.0

            u = np.clip(self.rng.random(current.shape), 1e-12, 1.0)
            sign = self.rng.choice([-1.0, 1.0], size=current.shape)
            quantum_step = sign * beta_q * np.abs(leader_centroid - current) * np.log(1.0 / u)
            new_position = attractor + quantum_step

            tunnel_mask = self.rng.random(current.shape) < self.tunnelling_probability * (1.0 - progress)
            tunnel_noise = self.rng.normal(
                loc=0.0,
                scale=0.15 * (1.0 - progress) + 0.02,
                size=current.shape,
            )
            new_position = np.where(tunnel_mask, alpha + tunnel_noise, new_position)
            updated_positions.append(np.clip(new_position, 0.0, 1.0))
        return np.asarray(updated_positions)

    def optimize(self, objective: Callable[[GATConfig], Dict[str, float]]) -> Tuple[GATConfig, float]:
        positions = self.rng.uniform(low=0.0, high=1.0, size=(self.wolves, self.dimensions))
        global_best_config: Optional[GATConfig] = None
        global_best_fitness = math.inf

        for iteration in range(self.iterations):
            scored = []
            for wolf_index, position in enumerate(positions):
                config = decode_position(position)
                if config not in self.cache:
                    self.cache[config] = objective(config)
                result = self.cache[config]
                fitness = float(result["fitness"])
                scored.append((fitness, position.copy(), config))
                self.search_log.append(
                    {
                        "iteration": iteration + 1,
                        "wolf": wolf_index + 1,
                        "fitness": fitness,
                        "validation_f1_macro": result["validation_f1_macro"],
                        "validation_accuracy": result["validation_accuracy"],
                        "parameters": result["parameters"],
                        **asdict(config),
                    }
                )
                if fitness < global_best_fitness:
                    global_best_fitness = fitness
                    global_best_config = config

            scored.sort(key=lambda item: item[0])
            alpha = scored[0][1]
            beta = scored[1][1]
            delta = scored[2][1]
            positions = self._quantum_update(positions, alpha, beta, delta, iteration)

        if global_best_config is None:
            raise RuntimeError("QGWO failed to evaluate any configuration")
        return global_best_config, global_best_fitness

## Plotting Utilities

In [10]:
def plot_class_balance(metadata: pd.DataFrame, output_path: Path, title: str) -> None:
    counts = metadata.groupby(["source", "label"]).size().unstack(fill_value=0)
    counts = counts.rename(columns={0: "Real", 1: "Fake"})
    ax = counts.plot(kind="bar", stacked=True, figsize=(10, 4), color=["#2E86AB", "#D1495B"])
    ax.set_title(title)
    ax.set_xlabel("Source")
    ax.set_ylabel("Records")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(output_path, dpi=220)
    plt.close()


def plot_training_history(history: List[Dict[str, float]], output_path: Path, title: str) -> None:
    frame = pd.DataFrame(history)
    plt.figure(figsize=(8, 4.5))
    plt.plot(frame["epoch"], frame["train_loss"], label="Train loss", color="#4C78A8")
    plt.plot(frame["epoch"], frame["validation_f1_macro"], label="Validation F1 macro", color="#F58518")
    plt.xlabel("Epoch")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=220)
    plt.close()


def plot_qgwo_convergence(search_log: pd.DataFrame, output_path: Path, title: str) -> None:
    best_by_iteration = search_log.groupby("iteration")["fitness"].min().cummin()
    plt.figure(figsize=(7, 4))
    plt.plot(best_by_iteration.index, best_by_iteration.values, marker="o", color="#54A24B")
    plt.xlabel("QGWO iteration")
    plt.ylabel("Best fitness")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(output_path, dpi=220)
    plt.close()


def plot_confusion(y_true: np.ndarray, y_pred: np.ndarray, output_path: Path, title: str) -> None:
    matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])
    plt.figure(figsize=(4.8, 4))
    plt.imshow(matrix, cmap="Blues")
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.xticks([0, 1], ["Real", "Fake"])
    plt.yticks([0, 1], ["Real", "Fake"])
    for row in range(matrix.shape[0]):
        for column in range(matrix.shape[1]):
            plt.text(column, row, str(matrix[row, column]), ha="center", va="center", color="#111111")
    plt.tight_layout()
    plt.savefig(output_path, dpi=220)
    plt.close()


def plot_roc(y_true: np.ndarray, y_prob: np.ndarray, output_path: Path, title: str) -> None:
    if len(np.unique(y_true)) < 2:
        return
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    plt.figure(figsize=(5.2, 4.2))
    plt.plot(fpr, tpr, color="#E45756", label=f"ROC-AUC = {auc:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--", color="#777777")
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    plt.title(title)
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(output_path, dpi=220)
    plt.close()


def plot_metric_comparison(metrics_frame: pd.DataFrame, output_path: Path) -> None:
    test = metrics_frame[metrics_frame["split"] == "test"].copy()
    if test.empty:
        return
    summary = test.groupby("experiment", as_index=False)["f1_macro"].mean()
    summary = summary.sort_values("f1_macro", ascending=True)
    plt.figure(figsize=(9, max(4, 0.35 * len(summary))))
    plt.barh(summary["experiment"], summary["f1_macro"], color="#4C78A8")
    plt.xlabel("Mean test F1 macro")
    plt.title("GAT + QGWO experiment comparison")
    plt.xlim(0, 1)
    plt.tight_layout()
    plt.savefig(output_path, dpi=240)
    plt.close()

## Experiment Runner

Each experiment produces:

- `metrics.csv`
- `qgwo_search_log.csv`
- `training_history.csv`
- `best_config.json`
- `test_predictions.csv`
- PNG figures

The final combined table is saved as `all_experiment_metrics.csv`.

In [11]:
REQUIRED_TABLE_COLUMNS = [
    "experiment",
    "seed",
    "split",
    "accuracy",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "precision_weighted",
    "recall_weighted",
    "f1_weighted",
    "roc_auc",
    "metric_type/note",
]


def load_experiment_bundle(experiment_name: str, seed: int) -> GraphBundle:
    spec = EXPERIMENTS[experiment_name]
    if spec["kind"] == "excel_graph":
        cache_path = OUTPUT_DIR / experiment_name / "excel_graph_cache.pt"
        tensors = ExcelGraphTensors(Path(spec["path"]), cache_path=cache_path)
        bundle = tensors.to_bundle(experiment_name)
        bundle.note = spec.get("note", "")
        return bundle

    source_names = spec["sources"]
    frame = load_sources(source_names)
    frame = sample_per_class(frame, MAX_ROWS_PER_CLASS, seed=seed)
    if frame["label"].nunique() < 2:
        raise ValueError(f"{experiment_name} has one class only after loading/sampling")
    note = f"Text graph; sources={','.join(source_names)}; {spec.get('note', '')}"
    return make_text_graph_bundle(
        name=experiment_name,
        frame=frame,
        config=TEXT_GRAPH_CONFIG,
        seed=seed,
        note=note,
    )


def run_experiment(experiment_name: str, seed: int) -> Dict[str, Any]:
    start_time = time.time()
    set_seed(seed)
    experiment_dir = OUTPUT_DIR / experiment_name / f"seed_{seed}"
    experiment_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n=== {experiment_name} | seed={seed} ===")
    bundle = load_experiment_bundle(experiment_name, seed=seed)
    train_indices = bundle.indices("train")
    validation_indices = bundle.indices("validation")
    test_indices = bundle.indices("test")
    if len(train_indices) == 0 or len(validation_indices) == 0 or len(test_indices) == 0:
        raise ValueError("Missing train/validation/test split")

    split_counts = pd.DataFrame(
        {
            "split": bundle.split,
            "label": bundle.y.numpy(),
            "source": bundle.metadata.get("source", pd.Series(["unknown"] * len(bundle.y))).to_numpy(),
        }
    )
    split_counts.to_csv(experiment_dir / "split_records.csv", index=False)
    bundle.metadata.to_csv(experiment_dir / "metadata.csv", index=False)
    plot_class_balance(bundle.metadata, experiment_dir / "class_balance.png", f"{experiment_name}: class balance")

    def loaders(loader_seed: int) -> Tuple[DataLoader, DataLoader, DataLoader]:
        return (
            make_loader(bundle, train_indices, BATCH_SIZE, True, loader_seed),
            make_loader(bundle, validation_indices, BATCH_SIZE, False, loader_seed),
            make_loader(bundle, test_indices, BATCH_SIZE, False, loader_seed),
        )

    train_loader, validation_loader, test_loader = loaders(seed)

    def objective(config: GATConfig) -> Dict[str, float]:
        config_seed = (
            seed
            + config.hidden_channels * 3
            + config.num_heads * 17
            + config.num_layers * 31
        )
        set_seed(config_seed)
        candidate_train, candidate_validation, _ = loaders(config_seed)
        model = JKDeepGATNet(
            num_features=bundle.num_features,
            hidden_channels=config.hidden_channels,
            num_heads=config.num_heads,
            num_layers=config.num_layers,
            dropout=config.dropout,
        )
        parameters = count_parameters(model)
        trained = train_model(
            model=model,
            bundle=bundle,
            train_loader=candidate_train,
            validation_loader=candidate_validation,
            train_indices=train_indices,
            device=DEVICE,
            epochs=CANDIDATE_EPOCHS,
            learning_rate=config.learning_rate,
            weight_decay=config.weight_decay,
            patience=CANDIDATE_EPOCHS,
            label_smoothing=config.label_smoothing,
        )
        metrics, _, _, _, _ = evaluate(trained.model, candidate_validation, DEVICE)
        fitness = 1.0 - metrics["f1_macro"] + COMPLEXITY_PENALTY * parameters
        print(
            f"candidate hidden={config.hidden_channels} heads={config.num_heads} "
            f"layers={config.num_layers} dropout={config.dropout:.3f} "
            f"lr={config.learning_rate:.6f} val_f1={metrics['f1_macro']:.4f} "
            f"fitness={fitness:.6f}"
        )
        return {
            "fitness": fitness,
            "validation_f1_macro": metrics["f1_macro"],
            "validation_accuracy": metrics["accuracy"],
            "parameters": parameters,
        }

    qgwo = QuantumGreyWolfOptimizer(wolves=WOLVES, iterations=QGWO_ITERATIONS, seed=seed)
    best_config, best_fitness = qgwo.optimize(objective)

    print("Best QGWO config:")
    print(json.dumps(asdict(best_config), indent=2))

    set_seed(seed)
    train_loader, validation_loader, test_loader = loaders(seed)
    final_model = JKDeepGATNet(
        num_features=bundle.num_features,
        hidden_channels=best_config.hidden_channels,
        num_heads=best_config.num_heads,
        num_layers=best_config.num_layers,
        dropout=best_config.dropout,
    )
    final_parameters = count_parameters(final_model)
    final_training = train_model(
        model=final_model,
        bundle=bundle,
        train_loader=train_loader,
        validation_loader=validation_loader,
        train_indices=train_indices,
        device=DEVICE,
        epochs=FINAL_EPOCHS,
        learning_rate=best_config.learning_rate,
        weight_decay=best_config.weight_decay,
        patience=PATIENCE,
        label_smoothing=best_config.label_smoothing,
    )

    metrics_rows: List[Dict[str, Any]] = []
    prediction_frames: List[pd.DataFrame] = []
    split_loaders = {
        "train": train_loader,
        "validation": validation_loader,
        "test": test_loader,
    }
    split_notes = {
        "train": "training split; fit diagnostic only",
        "validation": "model-selection split used by QGWO and early stopping",
        "test": "held-out split; primary paper result",
    }
    test_arrays = None
    for split_name, loader in split_loaders.items():
        metrics, y_true, y_pred, y_prob, row_indices = evaluate(final_training.model, loader, DEVICE)
        row = {
            "experiment": experiment_name,
            "seed": seed,
            "split": split_name,
            **metrics,
            "metric_type/note": f"GAT+QGWO binary classification; {split_notes[split_name]}",
        }
        metrics_rows.append(row)
        prediction_frames.append(
            pd.DataFrame(
                {
                    "split": split_name,
                    "row_index": row_indices,
                    "source_id": bundle.ids[row_indices],
                    "actual_label": y_true,
                    "predicted_label": y_pred,
                    "fake_probability": y_prob,
                }
            )
        )
        if split_name == "test":
            test_arrays = (y_true, y_pred, y_prob)

    metrics_frame = pd.DataFrame(metrics_rows)
    search_log = pd.DataFrame(qgwo.search_log)
    history_frame = pd.DataFrame(final_training.history)
    predictions_frame = pd.concat(prediction_frames, ignore_index=True)

    metrics_frame.to_csv(experiment_dir / "metrics.csv", index=False)
    search_log.to_csv(experiment_dir / "qgwo_search_log.csv", index=False)
    history_frame.to_csv(experiment_dir / "training_history.csv", index=False)
    predictions_frame.to_csv(experiment_dir / "predictions.csv", index=False)
    with open(experiment_dir / "best_config.json", "w", encoding="utf-8") as file:
        json.dump(
            {
                "experiment": experiment_name,
                "seed": seed,
                "best_config": asdict(best_config),
                "best_fitness": best_fitness,
                "best_epoch": final_training.best_epoch,
                "model_parameters": final_parameters,
                "bundle_note": bundle.note,
                "text_graph_config": asdict(TEXT_GRAPH_CONFIG),
            },
            file,
            indent=2,
        )
    torch.save(
        {
            "state_dict": final_training.model.state_dict(),
            "best_config": asdict(best_config),
            "num_features": bundle.num_features,
            "text_graph_config": asdict(TEXT_GRAPH_CONFIG),
        },
        experiment_dir / "gat_qgwo_model.pt",
    )

    plot_training_history(
        final_training.history,
        experiment_dir / "training_history.png",
        f"{experiment_name}: optimized GAT training",
    )
    plot_qgwo_convergence(
        search_log,
        experiment_dir / "qgwo_convergence.png",
        f"{experiment_name}: QGWO convergence",
    )
    if test_arrays is not None:
        y_true, y_pred, y_prob = test_arrays
        plot_confusion(y_true, y_pred, experiment_dir / "confusion_matrix_test.png", f"{experiment_name}: test confusion")
        plot_roc(y_true, y_prob, experiment_dir / "roc_curve_test.png", f"{experiment_name}: test ROC")

    elapsed = time.time() - start_time
    print(metrics_frame[REQUIRED_TABLE_COLUMNS].round(4).to_string(index=False))
    print(f"Finished in {elapsed / 60:.2f} minutes")

    return {
        "metrics": metrics_frame,
        "best_config": {
            "experiment": experiment_name,
            "seed": seed,
            "best_fitness": best_fitness,
            "best_epoch": final_training.best_epoch,
            "model_parameters": final_parameters,
            **asdict(best_config),
        },
        "search_log": search_log,
    }

## Run Experiments

This cell can take time. In `fast` mode it samples up to `MAX_ROWS_PER_CLASS` examples per class and uses a small QGWO budget. For final article results, switch to `RUN_MODE = "article"` in the configuration cell and restart the kernel.

In [12]:
all_metric_frames: List[pd.DataFrame] = []
all_config_rows: List[Dict[str, Any]] = []
all_search_logs: List[pd.DataFrame] = []
skipped: List[Dict[str, Any]] = []

for experiment_name in SELECTED_EXPERIMENTS:
    for seed in EXPERIMENT_SEEDS:
        try:
            result = run_experiment(experiment_name, seed=seed)
            all_metric_frames.append(result["metrics"])
            all_config_rows.append(result["best_config"])
            search_log = result["search_log"].copy()
            search_log.insert(0, "experiment", experiment_name)
            search_log.insert(1, "seed", seed)
            all_search_logs.append(search_log)
        except Exception as exc:
            print(f"SKIPPED {experiment_name} seed={seed}: {exc}")
            skipped.append({"experiment": experiment_name, "seed": seed, "reason": str(exc)})

metrics_all = pd.concat(all_metric_frames, ignore_index=True) if all_metric_frames else pd.DataFrame()
best_configs = pd.DataFrame(all_config_rows)
search_logs_all = pd.concat(all_search_logs, ignore_index=True) if all_search_logs else pd.DataFrame()
skipped_frame = pd.DataFrame(skipped)

if len(metrics_all):
    metrics_all.to_csv(OUTPUT_DIR / "all_experiment_metrics.csv", index=False)
    best_configs.to_csv(OUTPUT_DIR / "best_qgwo_configs.csv", index=False)
    search_logs_all.to_csv(OUTPUT_DIR / "all_qgwo_search_logs.csv", index=False)
    plot_metric_comparison(metrics_all, OUTPUT_DIR / "test_f1_macro_comparison.png")

if len(skipped_frame):
    skipped_frame.to_csv(OUTPUT_DIR / "skipped_experiments.csv", index=False)

print("Completed experiments:", metrics_all["experiment"].nunique() if len(metrics_all) else 0)
print("Skipped experiments:", len(skipped_frame))
display(skipped_frame)


=== prepared_excel_graphs | seed=42 ===
candidate hidden=24 heads=3 layers=3 dropout=0.399 lr=0.000102 val_f1=0.9933 fitness=0.007413
candidate hidden=24 heads=2 layers=2 dropout=0.235 lr=0.006901 val_f1=1.0000 fitness=0.000346
candidate hidden=16 heads=2 layers=2 dropout=0.082 lr=0.004176 val_f1=1.0000 fitness=0.000221
candidate hidden=12 heads=6 layers=3 dropout=0.439 lr=0.000169 val_f1=0.9566 fitness=0.044161
candidate hidden=24 heads=3 layers=1 dropout=0.120 lr=0.007407 val_f1=1.0000 fitness=0.000359
candidate hidden=16 heads=6 layers=2 dropout=0.337 lr=0.001719 val_f1=0.9967 fitness=0.004115
candidate hidden=32 heads=3 layers=2 dropout=0.050 lr=0.002209 val_f1=1.0000 fitness=0.000781
candidate hidden=32 heads=6 layers=2 dropout=0.170 lr=0.003857 val_f1=1.0000 fitness=0.001929
candidate hidden=32 heads=6 layers=1 dropout=0.106 lr=0.003939 val_f1=1.0000 fitness=0.001182
candidate hidden=24 heads=4 layers=2 dropout=0.085 lr=0.003391 val_f1=1.0000 fitness=0.000781
candidate hidden=32

""


## Article-Ready Tables

The first table has exactly the requested metric fields. The second table ranks experiments by held-out test F1 macro and ROC-AUC. If multiple seeds are used, the aggregate table reports mean and standard deviation.

In [13]:
if len(metrics_all):
    article_table = metrics_all[REQUIRED_TABLE_COLUMNS].copy()
    numeric_cols = [
        "accuracy",
        "precision_macro",
        "recall_macro",
        "f1_macro",
        "precision_weighted",
        "recall_weighted",
        "f1_weighted",
        "roc_auc",
    ]
    article_table[numeric_cols] = article_table[numeric_cols].round(6)
    display(article_table)
    article_table.to_csv(OUTPUT_DIR / "article_metrics_table.csv", index=False)

    test_rank = (
        article_table[article_table["split"] == "test"]
        .sort_values(["f1_macro", "roc_auc", "accuracy"], ascending=False)
        .reset_index(drop=True)
    )
    display(test_rank)
    test_rank.to_csv(OUTPUT_DIR / "test_split_ranked_results.csv", index=False)

    aggregate = (
        metrics_all[metrics_all["split"] == "test"]
        .groupby("experiment")[numeric_cols]
        .agg(["mean", "std"])
        .sort_values(("f1_macro", "mean"), ascending=False)
    )
    display(aggregate.round(6))
    aggregate.to_csv(OUTPUT_DIR / "test_split_aggregate_mean_std.csv")
else:
    print("No completed experiment metrics are available yet.")

,experiment,seed,split,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,roc_auc,metric_type/note
0,prepared_excel_graphs,42,train,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,GAT+QGWO binary classification; training split...
1,prepared_excel_graphs,42,validation,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,GAT+QGWO binary classification; model-selectio...
2,prepared_excel_graphs,42,test,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,GAT+QGWO binary classification; held-out split...
3,synthetic_politifact_all,42,train,0.948780,0.912903,0.931402,0.921741,0.950301,0.948780,0.949341,0.986987,GAT+QGWO binary classification; training split...
4,synthetic_politifact_all,42,validation,0.909091,0.842217,0.898923,0.865751,0.920721,0.909091,0.912558,0.961060,GAT+QGWO binary classification; model-selectio...
5,synthetic_politifact_all,42,test,0.910112,0.860720,0.860720,0.860720,0.910112,0.910112,0.910112,0.924883,GAT+QGWO binary classification; held-out split...
6,gossipcop_root,42,train,0.827976,0.829006,0.827976,0.827841,0.829006,0.827976,0.827841,0.912429,GAT+QGWO binary classification; training split...
7,gossipcop_root,42,validation,0.716667,0.718392,0.716667,0.716106,0.718392,0.716667,0.716106,0.779383,GAT+QGWO binary classification; model-selectio...
8,gossipcop_root,42,test,0.686111,0.686393,0.686111,0.685992,0.686393,0.686111,0.685992,0.762840,GAT+QGWO binary classification; held-out split...
9,politifact_new_folder,42,train,0.841570,0.838550,0.841014,0.839560,0.842576,0.841570,0.841857,0.932844,GAT+QGWO binary classification; training split...


,experiment,seed,split,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,roc_auc,metric_type/note
0,prepared_excel_graphs,42,test,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,GAT+QGWO binary classification; held-out split...
1,synthetic_politifact_all,42,test,0.910112,0.860720,0.860720,0.860720,0.910112,0.910112,0.910112,0.924883,GAT+QGWO binary classification; held-out split...
2,gossipcop_plus_plus_all,42,test,0.802778,0.804895,0.802778,0.802435,0.804895,0.802778,0.802435,0.855802,GAT+QGWO binary classification; held-out split...
3,politifact_new_folder,42,test,0.790541,0.789680,0.795015,0.789377,0.798282,0.790541,0.791492,0.868862,GAT+QGWO binary classification; held-out split...
4,politifact_plus_plus_all,42,test,0.786667,0.785206,0.800600,0.783550,0.810033,0.786667,0.789437,0.934033,GAT+QGWO binary classification; held-out split...
5,gossipcop_root,42,test,0.686111,0.686393,0.686111,0.685992,0.686393,0.686111,0.685992,0.762840,GAT+QGWO binary classification; held-out split...
6,all_text_sources,42,test,0.641667,0.641881,0.641667,0.641531,0.641881,0.641667,0.641531,0.709290,GAT+QGWO binary classification; held-out split...
7,factcheck_binary,42,test,0.613889,0.614174,0.613889,0.613647,0.614174,0.613889,0.613647,0.664923,GAT+QGWO binary classification; held-out split...


accuracy     precision_macro     recall_macro      \
                              mean std            mean std         mean std   
experiment                                                                    
prepared_excel_graphs     1.000000 NaN        1.000000 NaN     1.000000 NaN   
synthetic_politifact_all  0.910112 NaN        0.860720 NaN     0.860720 NaN   
gossipcop_plus_plus_all   0.802778 NaN        0.804895 NaN     0.802778 NaN   
politifact_new_folder     0.790541 NaN        0.789680 NaN     0.795015 NaN   
politifact_plus_plus_all  0.786667 NaN        0.785206 NaN     0.800600 NaN   
gossipcop_root            0.686111 NaN        0.686393 NaN     0.686111 NaN   
all_text_sources          0.641667 NaN        0.641881 NaN     0.641667 NaN   
factcheck_binary          0.613889 NaN        0.614174 NaN     0.613889 NaN   

                          f1_macro     precision_weighted     recall_weighted  \
                              mean std               mean std            mean   
experiment                                                                      
prepared_excel_graphs     1.000000 NaN           1.000000 NaN        1.000000   
synthetic_politifact_all  0.860720 NaN           0.910112 NaN        0.910112   
gossipcop_plus_plus_all   0.802435 NaN           0.804895 NaN        0.802778   
politifact_new_folder     0.789377 NaN           0.798282 NaN        0.790541   
politifact_plus_plus_all  0.783550 NaN           0.810033 NaN        0.786667   
gossipcop_root            0.685992 NaN           0.686393 NaN        0.686111   
all_text_sources          0.641531 NaN           0.641881 NaN        0.641667   
factcheck_binary          0.613647 NaN           0.614174 NaN        0.613889   

                             f1_weighted       roc_auc      
                         std        mean std      mean std  
experiment                                                  
prepared_excel_graphs    NaN    1.000000 NaN  1.000000 NaN  
synthetic_politifact_all NaN    0.910112 NaN  0.924883 NaN  
gossipcop_plus_plus_all  NaN    0.802435 NaN  0.855802 NaN  
politifact_new_folder    NaN    0.791492 NaN  0.868862 NaN  
politifact_plus_plus_all NaN    0.789437 NaN  0.934033 NaN  
gossipcop_root           NaN    0.685992 NaN  0.762840 NaN  
all_text_sources         NaN    0.641531 NaN  0.709290 NaN  
factcheck_binary         NaN    0.613647 NaN  0.664923 NaN

## Notes for Paper Writing

- Report the test split as the primary result.
- Keep validation results separate because QGWO uses validation F1 for model selection.
- Mention that text-only sources are converted to document graphs with sentence/chunk nodes and semantic-similarity edges.
- Mention that `half-true` PolitiFact verdicts are excluded for binary classification unless you explicitly define a three-class setup.
- For stronger publication evidence, use `RUN_MODE = "article"` with multiple seeds and report mean +/- standard deviation.
- Compare against a simple baseline in a separate table if reviewers ask for it, but keep this notebook's main model as GAT + QGWO.